In [11]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
fecha = '20260328'

Paraderos

In [13]:
#importar paradas

paradas = pd.read_csv('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Paraderos_Zonales_del_SITP.csv')

paradas.head(3)

,X,Y,objectid,cenefa,zona_sitp,nombre,via,direccion_bandera,localidad,longitud,latitud,consecutivo_zona,tipo_m_s,consola,panel,audio,zonas_nuevas,globalid,shape
0,1.001502e+06,1.010205e+06,1,001A00,00,C.C. Iserra 100,AC 100,AC 100 - KR 54,Barrios Unidos,-74.063971,4.688481,001,M,AC 100 - KR 54 (001A00),AC 100 - KR 54,Avenida Calle 100 Carrera 54,C,{1C0DBC4E-15BC-4BBE-BC16-5628077DBE2E},NaN
1,1.003505e+06,1.009719e+06,2,001A01,01,Br. Rincón del Chicó,AC 100,AC 100 - KR 13,Usaquén,-74.045914,4.684091,001,M,AC 100 - KR 13 (001A01),AC 100 - KR 13,Avenida Calle 100 Carrera 13,B,{60A22A44-AD56-4DF4-A3E6-6830B9FB0792},NaN
2,1.001238e+06,1.018098e+06,3,001A02,02,Gimnasio Iragua,AV. Boyacá,AV. Boyacá - AC 170,Suba,-74.066350,4.759867,001,S,AV. Boyacá - AC 170 (001A02),AV. Boyacá - AC 170,Avenida Boyacá Avenida Calle 170,C,{97155274-E4D5-45A4-891F-2DCA19FF10D9},NaN


Matriz de distancia

In [14]:
#Zonal

md_zonal = pd.read_csv(f'Z:/01 base_datos/06 matriz_distancia_FMS/{fecha}_matriz distancias.csv', encoding='latin')

#Troncal

md_troncal = pd.read_csv(f'Z:/01 base_datos/31 matriz_distancia_troncal_FMS/{fecha}_matriz_distancias_troncal.csv', encoding='latin')

md = pd.concat([md_zonal, md_troncal])

md.head(2)

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415.0,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415.0,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN


In [15]:
md = md[
    md["Id Nodo"].notna() &
    (md["Id Nodo"].astype(str).str.strip() != "") &
    (~md["Id Nodo"].astype(str).str.lower().isin(["nan"]))
]

md["Id Nodo"] = md["Id Nodo"].astype(int)
md["Id Ruta"] = md["Id Ruta"].astype(int)

md.head()

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN
3,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52442,110A05_TM,110A05_Br. Sabana del Dorado,686.0,594933.0,520819.0,NaN
4,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52369,070A05_TM,070A05_Br. Sabana del Dorado,964.0,595091.0,520593.0,NaN


In [16]:
#Cruzar datos con vd y md

md = md.rename(columns={
    "Id LÃ­nea": "Id Línea",
    "PosiciÃ³n": "Posición"
})

md.head(3)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2.0,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN


In [17]:
md = md.sort_values(
    by=['Id Línea', 'Id Ruta', 'Posición']
).reset_index(drop=True)

md.head(2)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."


In [18]:
md['orden'] = md.groupby(
    ['Id Línea', 'Id Ruta']
).cumcount() + 1

md.head(2)

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,orden
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",1
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",2


In [19]:
md["Parada"] = md["Etiqueta Nodo"].str.split("_").str[0]

md.head()

,ï»¿Tipo de Servicio,Id Línea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posición,Coordenada X,Coordenada Y,Atributos,orden,Parada
0,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71475,193B03,Av Suba - K114D,0.0,599402.0,525137.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",1,193B03
1,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,71476,195B03,Av Suba - K110A.,453.0,599813.0,524944.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",2,195B03
2,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61413,C004,21 Ãngeles A - 2,3097.0,601949.0,523468.0,"SinÃ³ptico, Punto de control (TM-GOAL)",3,C004
3,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61441,C032,Puentelargo A - 2,8326.0,603467.0,518807.0,SinÃ³ptico,4,C032
4,PADRON,10006,M84-C84,1,5.0,13.0,10013,M84 - C84_V1,Circular,61523,D054,Polo C - 2 Ã³ 5,11858.0,603726.0,516325.0,"SinÃ³ptico, Punto de control (TM-GOAL)",5,D054


Validaciones

In [ ]:
# ruta = f"C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_validacionZonal.csv"

# validaciones = pd.read_csv(ruta, low_memory=False)

# # Extrae el nombre del archivo automáticamente
# validaciones["Fecha_archivo"] = os.path.basename(ruta).split("_")[0]

# validaciones = validaciones[validaciones['Operador'] == '(005) GMOVIL ENGATIVA']

# validaciones.head(3)

In [21]:
ruta = f"Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/Validaciones_diarias/{fecha}_validacionZonal.csv"

validaciones = pd.read_csv(ruta, low_memory=False)

# Extrae el nombre del archivo automáticamente
validaciones["Fecha_archivo"] = os.path.basename(ruta).split("_")[0]

validaciones = validaciones[validaciones['Operador'] == '(005) GMOVIL ENGATIVA']

validaciones.head(3)

,Dispositivo,Emisor,Estacion_Parada,Fase,Fecha_Clearing,Fecha_Transaccion,Hora_Pico_SN,ID_Vehiculo,Linea,Nombre_Perfil,...,Operador,Ruta,Saldo_Despues_Transaccion,Saldo_Previo_a_Transaccion,Sistema,Tipo_Tarifa,Tipo_Tarjeta,Tipo_Vehiculo,Valor,Fecha_archivo
694100,220009010,(3101000) Bogota Card(Citizen),(52691) 181B05_TM|181B05_Br. La Estrada,Fase 3,2026-03-28,2026-03-28 03:57:10,Peak Time,507074,(10204) 402,(002) Adulto Mayor,...,(005) GMOVIL ENGATIVA,(12776) 402_V3,2400.0,5950.0,ZONAL,1,tullave Plus,(02) Urbano,3550.0,20260328
694101,220000828,(3101000) Bogota Card(Citizen),(52765) 204B05_TM|204B05_Br. San Antonio Engativá,Fase 3,2026-03-28,2026-03-28 03:57:54,Peak Time,504362,(10184) 740,(001) Anonymous,...,(005) GMOVIL ENGATIVA,(12774) 740_V2,2900.0,6450.0,ZONAL,1,tullave Básica,(02) Urbano,3550.0,20260328
694102,220000828,(3101000) Bogota Card(Citizen),(52765) 204B05_TM|204B05_Br. San Antonio Engativá,Fase 3,2026-03-28,2026-03-28 03:58:00,Peak Time,504362,(10184) 740,(001) Adulto,...,(005) GMOVIL ENGATIVA,(12774) 740_V2,0.0,3250.0,ZONAL,1,tullave Plus,(02) Urbano,3550.0,20260328


In [ ]:
validaciones = validaciones.drop([
    'Dispositivo', 'Emisor','Fase', 'Hora_Pico_SN','Nombre_Perfil', 
    'Operador', 'Saldo_Despues_Transaccion', 
    'Saldo_Previo_a_Transaccion', 'Sistema', 
    'Tipo_Tarifa', 'Tipo_Tarjeta','Valor'
], axis=1)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327


In [ ]:
# Convertir la columna 'Fecha_Transaccion' a tipo datetime
validaciones['Fecha_Transaccion'] = pd.to_datetime(validaciones['Fecha_Transaccion'])

# Extraer la parte de la hora y colocarla en una nueva columna 'Hora_Transaccion'
validaciones['Hora_Transaccion'] = validaciones['Fecha_Transaccion'].dt.time

# Mostrar el DataFrame resultante
validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02


In [ ]:
# Utilizando expresiones regulares para extraer el número entre paréntesis y el resto del texto
validaciones[['Numero_parada', 'Nombre_Completo']] = validaciones['Estacion_Parada'].str.extract(r'\((.*?)\)(.*)')

# Dividir la columna "Resto" en dos partes usando el carácter '|' como separador
validaciones[['Parada', 'Nombre']] = validaciones['Nombre_Completo'].str.split('|', expand=True)

# Mostrar el DataFrame resultante
validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,53727,070A08_TM|070A08_Br. Remanso,070A08_TM,070A08_Br. Remanso
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur


In [ ]:
# Utilizando expresiones regulares para extraer el número entre paréntesis y el resto del texto
validaciones[['Linea_1', 'Ruta_comercial']] = validaciones['Linea'].str.extract(r'\((.*?)\)(.*)')

validaciones[['Ruta_SAE', 'Sentido']] = validaciones['Ruta'].str.extract(r'\((.*?)\)(.*)')

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,53727,070A08_TM|070A08_Br. Remanso,070A08_TM,070A08_Br. Remanso,10690,DL219,12721,DL219_V2
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2


In [ ]:
validaciones["Hora_Transaccion"] = validaciones["Hora_Transaccion"].astype(str).str.strip()

# Convertir "nan" y vacíos a NaN reales
validaciones.loc[
    validaciones["Hora_Transaccion"].isin(["", "nan", "None"]),
    "Hora_Transaccion"
] = None

In [ ]:
validaciones["Franja"] = pd.to_datetime(
    validaciones["Hora_Transaccion"], errors="coerce"
).dt.hour

validaciones.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_32308\251133772.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  validaciones["Franja"] = pd.to_datetime(


,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,53727,070A08_TM|070A08_Br. Remanso,070A08_TM,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3


In [ ]:
validaciones["id_tarjeta"] = validaciones["Numero_Tarjeta"].astype("category").cat.codes

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,53727,070A08_TM|070A08_Br. Remanso,070A08_TM,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3,64640
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38747
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,131234
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,125393
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,53734,239A12_TM|239A12_Br. Palermo Sur,239A12_TM,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,48201


In [ ]:
validaciones["Parada"] = validaciones["Parada"].str.split("_").str[0]

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,53727,070A08_TM|070A08_Br. Remanso,070A08,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3,64640
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38747
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,131234
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,125393
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,48201


In [ ]:
#convertir a entero datos de linea y ruta

validaciones["Linea_1"] = validaciones["Linea_1"].astype(int)
validaciones["Ruta_SAE"] = validaciones["Ruta_SAE"].astype(int)
validaciones["Numero_parada"] = validaciones["Numero_parada"].astype(int)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,53727,070A08_TM|070A08_Br. Remanso,070A08,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3,64640
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38747
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,131234
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,125393
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,48201


In [ ]:
validaciones["Parada"] = validaciones["Parada"].astype(str).str.strip().str.upper()

validaciones = validaciones[validaciones["Parada"] != "(UNKNOWN)"]

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,Numero_parada,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,53727,070A08_TM|070A08_Br. Remanso,070A08,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3,64640
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38747
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,131234
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,125393
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,53734,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,48201


In [ ]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(linea, ruta, nodo):
    
    filtro = (
        (md['Id Línea'] == linea)&
        (md['Id Ruta'] == ruta)&
        (md['Id Nodo'] == nodo)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not md.loc[filtro].empty:
        # Obtener el primer valor
        mds = md.loc[filtro, 'orden'].iloc[0]
        return mds if not pd.isna(mds) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
validaciones['orden_parada'] = validaciones.apply(
    lambda row: calcular_turno(
        row['Linea_1'],
        row['Ruta_SAE'],
        row['Numero_parada']
    ),
    axis=1
)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Nombre_Completo,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,...,070A08_TM|070A08_Br. Remanso,070A08,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3,64640,52
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,...,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38747,23
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,...,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,131234,23
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,...,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,125393,23
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,...,239A12_TM|239A12_Br. Palermo Sur,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,48201,23


In [ ]:
paradas["cenefa"] = paradas["cenefa"].astype(str).str.strip().str.upper()


In [ ]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'latitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
validaciones['latitud'] = validaciones.apply(
    lambda row: calcular_turno(
        row['Parada']
    ),
    axis=1
)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Parada,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,...,070A08,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3,64640,52,4.607144
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,...,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38747,23,4.542074
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,...,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,131234,23,4.542074
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,...,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,125393,23,4.542074
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,...,239A12,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,48201,23,4.542074


In [ ]:
#Cruzar con datos de nodo y Numero_parada

def calcular_turno(parada):
    
    filtro = (
        (paradas['cenefa'] == parada)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not paradas.loc[filtro].empty:
        # Obtener el primer valor
        paradas1 = paradas.loc[filtro, 'longitud'].iloc[0]
        return paradas1 if not pd.isna(paradas1) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
validaciones['longitud'] = validaciones.apply(
    lambda row: calcular_turno(
        row['Parada']
    ),
    axis=1
)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Nombre,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud
770379,(53727) 070A08_TM|070A08_Br. Remanso,2026-03-27,2026-03-27 03:57:44,504162,(10690) DL219,7cac08012804edd980784e73782d1f6bdf713516c6cc54...,(12721) DL219_V2,(02) Urbano,20260327,03:57:44,...,070A08_Br. Remanso,10690,DL219,12721,DL219_V2,3,64640,52,4.607144,-74.117676
770380,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:49,504132,(10264) 614,4adf31c6c84cc9dcf400e31bb9c8b08599670c49ca6c5b...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:49,...,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,38747,23,4.542074,-74.112176
770381,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:54,504132,(10264) 614,fdc77ac6bff3ca0f2f93c6cd0206bc0843f86015d38009...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:54,...,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,131234,23,4.542074,-74.112176
770382,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:57:59,504132,(10264) 614,f26ebfc223d8234de313b2503a487cff7b8d46b127cf82...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:57:59,...,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,125393,23,4.542074,-74.112176
770383,(53734) 239A12_TM|239A12_Br. Palermo Sur,2026-03-27,2026-03-27 03:58:02,504132,(10264) 614,5d02cd0f49d30fe3dff4dd9862762ac84a6b730d44c893...,(12328) 614_Vuelta_V2,(02) Urbano,20260327,03:58:02,...,239A12_Br. Palermo Sur,10264,614,12328,614_Vuelta_V2,3,48201,23,4.542074,-74.112176


Matriz de origen - destino

In [ ]:
df_tarjetas = (
    validaciones
    .groupby([
        "Fecha_Clearing",
        "Numero_Tarjeta",
        "id_tarjeta",
        "Numero_parada",
        "Parada",
        "orden_parada",
        "latitud",
        "longitud",
        "Linea_1",
        "Ruta_comercial",
        "Ruta_SAE",
        "Sentido",
        "Franja"
    ])
    .size()
    .reset_index(name="conteo")
)

df_tarjetas.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
0,2026-03-27,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,52692,181C05,26,4.683280,-74.091076,10688,BD237,12758,BD237_V2,8,1
1,2026-03-27,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,56451,122A01,130,4.686592,-74.054870,10204,402,12776,402_V3,19,1
2,2026-03-27,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,1,52849,256A05,35,4.707867,-74.130065,10354,16-5,10674,16-5_V1,9,1
3,2026-03-27,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,2,57435,257A00,47,4.615328,-74.085103,10551,KL307,12856,KL307_V3,17,1
4,2026-03-27,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,3,53664,203A12,78,4.509112,-74.106413,10264,614,12327,614_Ida_V2,10,1


In [ ]:
df_tarjetas.groupby(["Numero_Tarjeta", "Franja"])["conteo"].sum()

df_tarjetas.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
0,2026-03-27,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,52692,181C05,26,4.683280,-74.091076,10688,BD237,12758,BD237_V2,8,1
1,2026-03-27,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,56451,122A01,130,4.686592,-74.054870,10204,402,12776,402_V3,19,1
2,2026-03-27,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,1,52849,256A05,35,4.707867,-74.130065,10354,16-5,10674,16-5_V1,9,1
3,2026-03-27,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,2,57435,257A00,47,4.615328,-74.085103,10551,KL307,12856,KL307_V3,17,1
4,2026-03-27,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,3,53664,203A12,78,4.509112,-74.106413,10264,614,12327,614_Ida_V2,10,1


In [ ]:
df_tarjetas.groupby(["Numero_Tarjeta", "Parada"])["conteo"].sum()

df_tarjetas.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
0,2026-03-27,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,52692,181C05,26,4.683280,-74.091076,10688,BD237,12758,BD237_V2,8,1
1,2026-03-27,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,56451,122A01,130,4.686592,-74.054870,10204,402,12776,402_V3,19,1
2,2026-03-27,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,1,52849,256A05,35,4.707867,-74.130065,10354,16-5,10674,16-5_V1,9,1
3,2026-03-27,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,2,57435,257A00,47,4.615328,-74.085103,10551,KL307,12856,KL307_V3,17,1
4,2026-03-27,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,3,53664,203A12,78,4.509112,-74.106413,10264,614,12327,614_Ida_V2,10,1


In [ ]:
origen = (
    df_tarjetas
    .sort_values(["Numero_Tarjeta", "conteo"], ascending=False)
    .drop_duplicates("Numero_Tarjeta")
)

origen.head()

,Fecha_Clearing,Numero_Tarjeta,id_tarjeta,Numero_parada,Parada,orden_parada,latitud,longitud,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,conteo
164134,2026-03-27,fffff6d484cfcd0551cc32b586251723708e54fe5601d1...,132421,57484,232B01,64,4.737600,-74.022752,10204,402,12776,402_V3,5,1
164133,2026-03-27,ffffdd041e9d48319b4cd2c4dd12e261862ff3903fddf1...,132420,56154,070A07,47,4.610841,-74.094863,10350,SE14,12739,SE14_Vuelta_V2,18,1
164132,2026-03-27,ffff88ba1d83a29ae02e8a871aca006f57d24cd3dda424...,132419,57683,462A00,57,4.597918,-74.071928,10551,KL307,12856,KL307_V3,18,1
164131,2026-03-27,fffea3f13cccfae58ac3a9f6a89d3b24f9467cf93fe7ff...,132418,51088,016B09,46,4.607645,-74.141830,10304,806,12780,806_Vuelta_V3,14,1
164130,2026-03-27,fffe8fac6be43f87132c6ed2bd2469994e3a3dd4ab98f4...,132417,54723,211B03,29,4.696515,-74.084829,10339,C25,12756,C25_V3,14,1


In [ ]:
validaciones["Hora_dt"] = pd.to_datetime(validaciones["Hora_Transaccion"], errors="coerce")

od = (
    validaciones
    .sort_values(["Numero_Tarjeta","id_tarjeta", "Hora_dt"])
    .groupby(["Numero_Tarjeta", "id_tarjeta","Ruta_comercial", "Sentido"])
    .agg(
        origen_parada=("Parada", "first"),
        destino_parada=("Parada", "last"),
        origen_hora=("Hora_dt", "first"),
        destino_hora=("Hora_dt", "last")
    )
    .reset_index()
)

od.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_32308\2080183682.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  validaciones["Hora_dt"] = pd.to_datetime(validaciones["Hora_Transaccion"], errors="coerce")


,Numero_Tarjeta,id_tarjeta,Ruta_comercial,Sentido,origen_parada,destino_parada,origen_hora,destino_hora
0,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,402,402_V3,122A01,122A01,2026-04-06 19:13:41,2026-04-06 19:13:41
1,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,0,BD237,BD237_V2,181C05,181C05,2026-04-06 08:38:12,2026-04-06 08:38:12
2,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,1,16-5,16-5_V1,256A05,256A05,2026-04-06 09:53:49,2026-04-06 09:53:49
3,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,2,KL307,KL307_V3,257A00,257A00,2026-04-06 17:18:33,2026-04-06 17:18:33
4,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,3,614,614_Ida_V2,203A12,203A12,2026-04-06 10:56:55,2026-04-06 10:56:55


In [ ]:
validaciones = validaciones.sort_values(["id_tarjeta", "Hora_dt"])

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Linea_1,Ruta_comercial,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt
823104,(52692) 181C05_TM|181C05_Br. La Estrada,2026-03-27,2026-03-27 08:38:12,502133,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260327,08:38:12,...,10688,BD237,12758,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-06 08:38:12
924379,(56451) 122A01_TM|122A01_Autopista Norte,2026-03-27,2026-03-27 19:13:41,504186,(10204) 402,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12776) 402_V3,(02) Urbano,20260327,19:13:41,...,10204,402,12776,402_V3,19,0,130,4.686592,-74.054870,2026-04-06 19:13:41
834321,(52849) 256A05_TM|256A05_Br. La Riviera,2026-03-27,2026-03-27 09:53:49,507135,(10354) 16-5,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,(10674) 16-5_V1,(02) Urbano,20260327,09:53:49,...,10354,16-5,10674,16-5_V1,9,1,35,4.707867,-74.130065,2026-04-06 09:53:49
901693,(57435) 257A00_TM|257A00_Pl. Paloquemao,2026-03-27,2026-03-27 17:18:33,502163,(10551) KL307,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,(12856) KL307_V3,(02) Urbano,20260327,17:18:33,...,10551,KL307,12856,KL307_V3,17,2,47,4.615328,-74.085103,2026-04-06 17:18:33
842427,(53664) 203A12_TM|203A12_Br. Gran Yomasa I,2026-03-27,2026-03-27 10:56:55,504182,(10264) 614,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,(12327) 614_Ida_V2,(02) Urbano,20260327,10:56:55,...,10264,614,12327,614_Ida_V2,10,3,78,4.509112,-74.106413,2026-04-06 10:56:55


In [ ]:
validaciones["Parada_siguiente"] = validaciones.groupby("id_tarjeta")["Parada"].shift(-1)
validaciones["Hora_siguiente"] = validaciones.groupby("id_tarjeta")["Hora_dt"].shift(-1)

validaciones.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt,Parada_siguiente,Hora_siguiente
823104,(52692) 181C05_TM|181C05_Br. La Estrada,2026-03-27,2026-03-27 08:38:12,502133,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260327,08:38:12,...,12758,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-06 08:38:12,122A01,2026-04-06 19:13:41
924379,(56451) 122A01_TM|122A01_Autopista Norte,2026-03-27,2026-03-27 19:13:41,504186,(10204) 402,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12776) 402_V3,(02) Urbano,20260327,19:13:41,...,12776,402_V3,19,0,130,4.686592,-74.054870,2026-04-06 19:13:41,NaN,NaT
834321,(52849) 256A05_TM|256A05_Br. La Riviera,2026-03-27,2026-03-27 09:53:49,507135,(10354) 16-5,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,(10674) 16-5_V1,(02) Urbano,20260327,09:53:49,...,10674,16-5_V1,9,1,35,4.707867,-74.130065,2026-04-06 09:53:49,NaN,NaT
901693,(57435) 257A00_TM|257A00_Pl. Paloquemao,2026-03-27,2026-03-27 17:18:33,502163,(10551) KL307,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,(12856) KL307_V3,(02) Urbano,20260327,17:18:33,...,12856,KL307_V3,17,2,47,4.615328,-74.085103,2026-04-06 17:18:33,NaN,NaT
842427,(53664) 203A12_TM|203A12_Br. Gran Yomasa I,2026-03-27,2026-03-27 10:56:55,504182,(10264) 614,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,(12327) 614_Ida_V2,(02) Urbano,20260327,10:56:55,...,12327,614_Ida_V2,10,3,78,4.509112,-74.106413,2026-04-06 10:56:55,NaN,NaT


In [ ]:
od = validaciones[
    validaciones["Parada"] != validaciones["Parada_siguiente"]
].copy()

od.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Ruta_SAE,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt,Parada_siguiente,Hora_siguiente
823104,(52692) 181C05_TM|181C05_Br. La Estrada,2026-03-27,2026-03-27 08:38:12,502133,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260327,08:38:12,...,12758,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-06 08:38:12,122A01,2026-04-06 19:13:41
924379,(56451) 122A01_TM|122A01_Autopista Norte,2026-03-27,2026-03-27 19:13:41,504186,(10204) 402,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12776) 402_V3,(02) Urbano,20260327,19:13:41,...,12776,402_V3,19,0,130,4.686592,-74.054870,2026-04-06 19:13:41,NaN,NaT
834321,(52849) 256A05_TM|256A05_Br. La Riviera,2026-03-27,2026-03-27 09:53:49,507135,(10354) 16-5,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,(10674) 16-5_V1,(02) Urbano,20260327,09:53:49,...,10674,16-5_V1,9,1,35,4.707867,-74.130065,2026-04-06 09:53:49,NaN,NaT
901693,(57435) 257A00_TM|257A00_Pl. Paloquemao,2026-03-27,2026-03-27 17:18:33,502163,(10551) KL307,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,(12856) KL307_V3,(02) Urbano,20260327,17:18:33,...,12856,KL307_V3,17,2,47,4.615328,-74.085103,2026-04-06 17:18:33,NaN,NaT
842427,(53664) 203A12_TM|203A12_Br. Gran Yomasa I,2026-03-27,2026-03-27 10:56:55,504182,(10264) 614,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,(12327) 614_Ida_V2,(02) Urbano,20260327,10:56:55,...,12327,614_Ida_V2,10,3,78,4.509112,-74.106413,2026-04-06 10:56:55,NaN,NaT


In [ ]:
map_orden = od.drop_duplicates(subset=['Ruta_comercial', 'Parada']) \
    .set_index(['Ruta_comercial', 'Parada'])['orden_parada']

In [ ]:
od['orden_destino'] = od.set_index(['Ruta_comercial', 'Parada_siguiente']).index.map(map_orden)

od.head()

,Estacion_Parada,Fecha_Clearing,Fecha_Transaccion,ID_Vehiculo,Linea,Numero_Tarjeta,Ruta,Tipo_Vehiculo,Fecha_archivo,Hora_Transaccion,...,Sentido,Franja,id_tarjeta,orden_parada,latitud,longitud,Hora_dt,Parada_siguiente,Hora_siguiente,orden_destino
823104,(52692) 181C05_TM|181C05_Br. La Estrada,2026-03-27,2026-03-27 08:38:12,502133,(10688) BD237,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12758) BD237_V2,(02) Urbano,20260327,08:38:12,...,BD237_V2,8,0,26,4.683280,-74.091076,2026-04-06 08:38:12,122A01,2026-04-06 19:13:41,67.0
924379,(56451) 122A01_TM|122A01_Autopista Norte,2026-03-27,2026-03-27 19:13:41,504186,(10204) 402,00004c56f2824e06b7f62267528fda6cbb74f9e4118198...,(12776) 402_V3,(02) Urbano,20260327,19:13:41,...,402_V3,19,0,130,4.686592,-74.054870,2026-04-06 19:13:41,NaN,NaT,NaN
834321,(52849) 256A05_TM|256A05_Br. La Riviera,2026-03-27,2026-03-27 09:53:49,507135,(10354) 16-5,0000e974dfab018e295764617fa6d5cb790c24fe99bd7e...,(10674) 16-5_V1,(02) Urbano,20260327,09:53:49,...,16-5_V1,9,1,35,4.707867,-74.130065,2026-04-06 09:53:49,NaN,NaT,NaN
901693,(57435) 257A00_TM|257A00_Pl. Paloquemao,2026-03-27,2026-03-27 17:18:33,502163,(10551) KL307,00018763caf4eed5e9342f390f39fa6230fffd72f498b3...,(12856) KL307_V3,(02) Urbano,20260327,17:18:33,...,KL307_V3,17,2,47,4.615328,-74.085103,2026-04-06 17:18:33,NaN,NaT,NaN
842427,(53664) 203A12_TM|203A12_Br. Gran Yomasa I,2026-03-27,2026-03-27 10:56:55,504182,(10264) 614,0001d83d5bb0928c374887aa8dbece1e675f93524ff15d...,(12327) 614_Ida_V2,(02) Urbano,20260327,10:56:55,...,614_Ida_V2,10,3,78,4.509112,-74.106413,2026-04-06 10:56:55,NaN,NaT,NaN


In [ ]:
od_final = od[[
    "id_tarjeta",
    "Parada",
    "orden_parada",
    "Hora_dt",
    "Parada_siguiente",
    "orden_destino",
    "Hora_siguiente",
    "Ruta_comercial",
    "ID_Vehiculo",
    "Sentido",
    "Franja",
    "latitud",
    "longitud"
]].rename(columns={
    "Parada": "origen_parada",
    "Parada_siguiente": "destino_parada",
    "Hora_dt": "origen_hora",
    "Hora_siguiente": "destino_hora"
})

od_final.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud
823104,0,181C05,26,2026-04-06 08:38:12,122A01,67.0,2026-04-06 19:13:41,BD237,502133,BD237_V2,8,4.683280,-74.091076
924379,0,122A01,130,2026-04-06 19:13:41,NaN,NaN,NaT,402,504186,402_V3,19,4.686592,-74.054870
834321,1,256A05,35,2026-04-06 09:53:49,NaN,NaN,NaT,16-5,507135,16-5_V1,9,4.707867,-74.130065
901693,2,257A00,47,2026-04-06 17:18:33,NaN,NaN,NaT,KL307,502163,KL307_V3,17,4.615328,-74.085103
842427,3,203A12,78,2026-04-06 10:56:55,NaN,NaN,NaT,614,504182,614_Ida_V2,10,4.509112,-74.106413


In [ ]:
od_finalp = od_final.sort_values([
    'Ruta_comercial',
    'Sentido',
    'ID_Vehiculo',
    'Franja',
    'origen_hora'
])

od_finalp.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud
836488,58448,441A06,9,2026-04-06 10:09:36,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649
836499,12463,441A06,9,2026-04-06 10:09:41,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649
836509,35702,441A06,9,2026-04-06 10:09:46,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649
836519,118804,441A06,9,2026-04-06 10:09:53,124A06,191.0,2026-04-06 15:35:38,12,502007,12,10,4.687194,-74.155649
836528,49171,441A06,9,2026-04-06 10:09:58,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649


In [ ]:
od_finalp['usuario_desciende'] = od_finalp['destino_parada'].notna().astype(int)

od_finalp.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud,usuario_desciende
836488,58448,441A06,9,2026-04-06 10:09:36,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0
836499,12463,441A06,9,2026-04-06 10:09:41,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0
836509,35702,441A06,9,2026-04-06 10:09:46,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0
836519,118804,441A06,9,2026-04-06 10:09:53,124A06,191.0,2026-04-06 15:35:38,12,502007,12,10,4.687194,-74.155649,1
836528,49171,441A06,9,2026-04-06 10:09:58,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0


In [ ]:
od_finalp['sube'] = 1
od_finalp['baja'] = od_finalp['usuario_desciende']

od_finalp.head()

,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud,usuario_desciende,sube,baja
836488,58448,441A06,9,2026-04-06 10:09:36,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0
836499,12463,441A06,9,2026-04-06 10:09:41,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0
836509,35702,441A06,9,2026-04-06 10:09:46,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0
836519,118804,441A06,9,2026-04-06 10:09:53,124A06,191.0,2026-04-06 15:35:38,12,502007,12,10,4.687194,-74.155649,1,1,1
836528,49171,441A06,9,2026-04-06 10:09:58,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0


In [ ]:
od_finalp['conteo_pasajeros'] = od_finalp.groupby(
    ['Ruta_comercial', 'Sentido', 'ID_Vehiculo', 'Franja']
).apply(
    lambda x: (x['sube'] - x['baja']).cumsum()
).reset_index(level=[0,1,2,3], drop=True)

od_finalp.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_32308\2394391313.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


,id_tarjeta,origen_parada,orden_parada,origen_hora,destino_parada,orden_destino,destino_hora,Ruta_comercial,ID_Vehiculo,Sentido,Franja,latitud,longitud,usuario_desciende,sube,baja,conteo_pasajeros
836488,58448,441A06,9,2026-04-06 10:09:36,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0,1
836499,12463,441A06,9,2026-04-06 10:09:41,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0,2
836509,35702,441A06,9,2026-04-06 10:09:46,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0,3
836519,118804,441A06,9,2026-04-06 10:09:53,124A06,191.0,2026-04-06 15:35:38,12,502007,12,10,4.687194,-74.155649,1,1,1,3
836528,49171,441A06,9,2026-04-06 10:09:58,NaN,NaN,NaT,12,502007,12,10,4.687194,-74.155649,0,1,0,4


In [ ]:
validaciones.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_validaciones_procesadas.csv', index= False, sep=';')

In [ ]:
od.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_origen_destino.csv', index= False, sep=';')

In [ ]:
df_tarjetas.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_transaciones_tarjeta.csv', index= False, sep=';')

In [ ]:
od_final.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_origen_destino_paradas.csv', index= False, sep=';')

In [ ]:
od_finalp.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Validaciones/{fecha}_origen_destino_pax.csv', index= False, sep=';')